In [1]:
## Import divers
import pandas as pd
import matplotlib as plot

FIXED_SEED = 67

In [2]:
## Chargement observation des datas
df = pd.read_csv("../data/raw/data.csv")

In [ ]:
## Aperçu
df.head()

In [4]:
## Dimensions 
df.shape

(4657, 4)

In [ ]:
## Infos
df.info()

In [ ]:
## Stats
df.describe()

In [ ]:
## Valeurs manquantes
df.isnull().sum()

In [ ]:
## Répartition des genres
print((df["genre"].value_counts(normalize=True) * 100).round(2))
df["genre"].value_counts().plot(kind="bar")

In [ ]:
## Doublons
df.duplicated().sum()
df.duplicated(subset=["summary"]).sum()
duplicates = df[df.duplicated(
    subset=["summary"],
    keep=False
)]

duplicates.head(20)

In [ ]:
## Doublons v2
duplicates.groupby(["title", "summary"])["genre"].nunique().value_counts()

In [ ]:
## Nombre de livres avec plusieurs genres
conflicting = duplicates[
    duplicates.groupby(["summary"])["genre"].transform("nunique") > 1
]

conflicting.groupby(
    ["summary"]
)["genre"].nunique().value_counts()

In [ ]:
## Nombre de résumés vides
(df["summary"].str.strip() == "").sum()

In [ ]:
## Stats longueur des résumés
df["summary"].str.len().describe()

In [ ]:
## Afficher les 10 résumés les plus longs
df["summary_length"] = df["summary"].str.len()
df.nlargest(15,"summary_length")[["genre", "summary_length", "summary"]]

In [ ]:
## Afficher les 10 résumés les plus courts
df["summary_length"] = df["summary"].str.len()
df.nsmallest(15,"summary_length")[["genre", "summary_length", "summary"]]

In [ ]:
## Afficher les résumés de - de n caractères
n = 50
short_summaries = df[df["summary"].str.len() < n]

short_summaries.sort_values("summary", key=lambda x: x.str.len())[
    ["genre", "summary"]
]

In [10]:
## Identification des patterns douteux et les retirer
bad_patterns = [
    r"==Receptio",
    r"==Character",
    r"- --&#62; &#60;!-",
    r"~Plot outline description~",
    r"~Plot outline description",
    r"-->",
    r"<!--"
]

for pattern in bad_patterns:
    df["summary"] = df["summary"].str.replace(
        pattern, "", regex=True
    )
df = df[df["summary"] != "-"]

In [125]:
## Vérification de la légitimité de commentaires possédant du bruit

result = df[df["summary"].str.contains(
    "Plot outline|==Receptio|==Character",
    case=False,
    na=False
)][["summary"]]

for summary in result["summary"]:
    print(repr(summary))

In [8]:
## Suppression des résumés vides
df["summary"] = df["summary"].str.strip()
df = df[df["summary"] != ""]

In [9]:
## Suppression colonnes index et titre
df = df.drop(columns=["index","title"])

In [12]:
## Suppression des doublons exactes (même résumé + même genre)
## On conserve les livres avec plusieurs genres différents
df = df.drop_duplicates(
    subset=["summary", "genre"]
)

In [ ]:
## Vérification suppression des doublons exactes
df.duplicated(subset=["summary", "genre"]).sum()

In [15]:
## Regrouper les livres avec plusieurs genres sur une seul ligne 
df = (
    df.groupby(["summary"], as_index=False)["genre"]
      .agg(list)
)
df.shape

(4535, 2)

In [ ]:
## Nombre de livres avec plusieurs genres
df["genre"].str.len().value_counts()

In [ ]:
## Nombre de livres uniques
df[["summary"]].drop_duplicates().shape

In [ ]:
## Afficher les livres multi-genres
df[df["genre"].str.len() > 1]

In [19]:
## Enregistrement du dataset
df.to_csv("../data/processed/data_clean.csv", index=False)